# Task 2.2 — Decision tree

Secondo dei due classificatori manuali. ID3 con information gain. Motivazioni:
`documentation/02.2_decision_tree.md`.

In [1]:
import numpy as np
import pandas as pd

SEME = 42          # coerente con il Task 1

manuale = pd.read_csv("../data/manuale.csv").set_index("USERID")
y = manuale["ABBANDONO"]
X = manuale.drop(columns="ABBANDONO")

print("campioni:", len(manuale), " feature:", X.shape[1])
print("distribuzione delle classi:", y.value_counts().to_dict())
manuale

campioni: 12  feature: 8
distribuzione delle classi: {0: 6, 1: 6}


,n_azioni,n_attivita_distinte,n_giorni_attivi,durata_giorni,feature0_media,feature1_media,feature2_media,feature3_media,ABBANDONO
USERID,,,,,,,,,
3969,44,26,5,4.941,-0.167,0.258,-0.053,-0.013,0
635,138,73,7,26.158,0.081,0.228,-0.093,-0.044,0
4344,5,3,1,0.000,-0.320,-0.436,0.107,-0.067,1
1714,97,40,11,27.117,0.014,-0.200,-0.012,-0.057,0
1123,8,5,1,0.013,-0.320,-0.118,0.044,-0.067,1
3637,6,5,1,0.082,-0.320,-0.012,0.023,-0.067,1
4413,94,19,5,5.156,-0.191,-0.409,0.096,-0.031,1
2535,65,42,6,25.987,1.215,-0.436,0.045,0.831,0
4200,7,5,2,3.618,-0.320,0.291,0.035,-0.067,1


## 1. Entropia del nodo radice

6 abbandoni e 6 non-abbandoni: attendiamo 1 bit esatto, cioe' massima incertezza.

In [2]:
def entropia_binaria(positivi, totali):
    """Entropia di Shannon in bit, calcolata dai *conteggi* invece che dalle etichette.

    Scriverla sui conteggi la rende un'espressione di soli operatori aritmetici: NumPy la
    applica elemento per elemento, quindi la stessa riga vale per un nodo solo o per mille
    nodi in parallelo. Al punto 3 la useremo per valutare tutte le soglie candidate insieme.
    """
    positivi, totali = np.asarray(positivi, float), np.asarray(totali, float)
    p = np.divide(positivi, totali, out=np.zeros_like(totali), where=totali > 0)
    q = 1 - p
    with np.errstate(divide="ignore", invalid="ignore"):
        h = -(np.where(p > 0, p * np.log2(p), 0.0) + np.where(q > 0, q * np.log2(q), 0.0))
    return h + 0.0                                   # il + 0.0 evita il -0.0 sui nodi puri


def entropia(etichette):
    """Entropia di un vettore di etichette binarie, in bit."""
    return float(entropia_binaria((etichette == 1).sum(), len(etichette)))


H_radice = entropia(y)
print(f"entropia della radice = {H_radice:.4f} bit")

entropia della radice = 1.0000 bit


## 2. Come si divide una feature continua

Le nostre feature sono continue: un ramo per valore darebbe un campione per foglia. Usiamo soglie
sui **punti medi fra valori consecutivi** — con *n* valori distinti bastano *n−1* prove.

E' quello che fa scikit-learn: `DecisionTreeClassifier` non accetta variabili categoriche.

In [3]:
def soglie_candidate(colonna):
    """Punti medi fra valori consecutivi distinti: gli unici tagli che cambiano la partizione."""
    v = np.unique(colonna)
    return (v[:-1] + v[1:]) / 2

print("n_giorni_attivi   valori distinti:", np.unique(X["n_giorni_attivi"]))
print("                  soglie da provare:", soglie_candidate(X["n_giorni_attivi"]))
print()
print("soglie da provare per ciascuna feature (= valori distinti - 1):")
print(X.nunique().sub(1).to_string())

n_giorni_attivi   valori distinti: [ 1  2  5  6  7  9 11]
                  soglie da provare: [ 1.5  3.5  5.5  6.5  8.  10. ]

soglie da provare per ciascuna feature (= valori distinti - 1):
n_azioni               11
n_attivita_distinte     9
n_giorni_attivi         6
durata_giorni          11
feature0_media          8
feature1_media         10
feature2_media         11
feature3_media          7


## 3. Information gain di ogni feature

In [4]:
def guadagno(a_sinistra, y):
    """Information gain di una o piu' partizioni, date come colonne di una matrice booleana.

    `a_sinistra` ha una riga per campione e una colonna per split candidato. I conteggi
    delle due classi in ciascun ramo escono da due somme di colonna, e l'entropia da
    `entropia_binaria`: tutti gli split vengono valutati insieme, senza cicli.
    """
    positivi = (y.to_numpy() == 1)[:, None]
    n = len(y)
    n_sx   = a_sinistra.sum(axis=0)                     # campioni nel ramo "si", per split
    pos_sx = (a_sinistra & positivi).sum(axis=0)        # di cui abbandoni
    return (entropia(y)
            - (n_sx / n)       * entropia_binaria(pos_sx, n_sx)
            - ((n - n_sx) / n) * entropia_binaria(int(positivi.sum()) - pos_sx, n - n_sx))

def classifica_feature(X, y):
    """Per ogni feature la soglia migliore e il guadagno che ottiene.

    Tutte le coppie (feature, soglia) del nodo sono valutate in un'unica passata: le soglie
    candidate finiscono impilate in un solo vettore e `X[:, quale] <= soglie` costruisce in
    un colpo la matrice campioni x coppie di tutte le partizioni possibili. L'unico ciclo
    rimasto scorre gli 8 *nomi* di colonna, perche' ogni feature ha un numero diverso di
    soglie e le liste non stanno in un unico array; sui dati non si itera mai.
    """
    candidate = [soglie_candidate(X[c]) for c in X.columns]
    quale  = np.repeat(np.arange(X.shape[1]), [len(s) for s in candidate])
    soglie = np.concatenate(candidate)

    tutte = pd.DataFrame({"feature": np.asarray(X.columns)[quale],
                          "soglia_migliore": soglie,
                          "information_gain": guadagno(X.to_numpy()[:, quale] <= soglie, y)})
    migliori = tutte.loc[tutte.groupby("feature", sort=False)["information_gain"].idxmax()]
    return (migliori.sort_values("information_gain", ascending=False, kind="stable")
                    .reset_index(drop=True))

classifica_feature(X, y).round(4)          # la soglia esatta resta quella non arrotondata

,feature,soglia_migliore,information_gain
0,n_attivita_distinte,22.500,1.0000
1,n_azioni,31.500,0.6549
2,n_giorni_attivi,3.500,0.6549
3,durata_giorni,4.813,0.6549
4,feature0_media,-0.179,0.6549
5,feature3_media,-0.062,0.6549
6,feature2_media,-0.039,0.4591
7,feature1_media,0.082,0.1957


`n_attivita_distinte` ottiene **IG = 1,0000**, il massimo: separa i 12 campioni senza errori.

Le cinque feature successive pareggiano a 0,6549 — misurano tutte la lunghezza della storia.

In [5]:
radice = classifica_feature(X, y).iloc[0]
FEATURE, SOGLIA = radice["feature"], radice["soglia_migliore"]

sinistra = manuale[X[FEATURE] <= SOGLIA].sort_values(FEATURE)
destra   = manuale[X[FEATURE] >  SOGLIA].sort_values(FEATURE)

print(f"split scelto: {FEATURE} <= {SOGLIA:.3f}\n")
print(f"ramo SI  ({len(sinistra):2d} campioni): {FEATURE} = {list(sinistra[FEATURE])}")
print(f"         etichette = {list(sinistra['ABBANDONO'])}  ->  entropia {entropia(sinistra['ABBANDONO']):.4f}")
print(f"ramo NO  ({len(destra):2d} campioni): {FEATURE} = {list(destra[FEATURE])}")
print(f"         etichette = {list(destra['ABBANDONO'])}  ->  entropia {entropia(destra['ABBANDONO']):.4f}")

split scelto: n_attivita_distinte <= 22.500

ramo SI  ( 6 campioni): n_attivita_distinte = [3, 5, 5, 5, 9, 19]
         etichette = [1, 1, 1, 1, 1, 1]  ->  entropia 0.0000
ramo NO  ( 6 campioni): n_attivita_distinte = [26, 40, 42, 59, 62, 73]
         etichette = [0, 0, 0, 0, 0, 0]  ->  entropia 0.0000


## 4. L'albero risultante

Entrambi i figli sono puri: l'algoritmo si ferma da solo. Un nodo, due foglie.

```
n_attivita_distinte <= 22.5 ?
├── si  -> ABBANDONO = 1   (6 campioni)
└── no  -> ABBANDONO = 0   (6 campioni)
```

Non abbiamo applicato pruning: la profondita' massima nel codice e' 3 ma non entra mai in gioco.

In [6]:
def cresci(X, y, profondita=0, max_profondita=3):
    """ID3 con soglie numeriche: sceglie ricorsivamente lo split con guadagno massimo."""
    if entropia(y) == 0 or profondita == max_profondita:
        return {"foglia": int(y.mode().iloc[0]), "n": len(y)}
    c = classifica_feature(X, y).iloc[0]
    if c["information_gain"] <= 1e-12:
        return {"foglia": int(y.mode().iloc[0]), "n": len(y)}
    maschera = X[c["feature"]] <= c["soglia_migliore"]
    return {"feature": c["feature"], "soglia": c["soglia_migliore"], "ig": c["information_gain"], "n": len(y),
            "si": cresci(X[maschera],  y[maschera],  profondita + 1, max_profondita),
            "no": cresci(X[~maschera], y[~maschera], profondita + 1, max_profondita)}

def stampa(nodo, indent=""):
    if "foglia" in nodo:
        print(f"{indent}=> ABBANDONO = {nodo['foglia']}   ({nodo['n']} campioni)")
        return
    print(f"{indent}{nodo['feature']} <= {nodo['soglia']:.3f} ?   IG={nodo['ig']:.4f}   ({nodo['n']} campioni)")
    print(f"{indent}  si:"); stampa(nodo["si"], indent + "     ")
    print(f"{indent}  no:"); stampa(nodo["no"], indent + "     ")

def predici(radice, X):
    """Predizioni per tutte le righe insieme, scendendo l'albero a maschere.

    Invece di ripercorrere l'albero riga per riga, ogni nodo riceve la maschera booleana
    dei campioni che lo raggiungono e la divide in due con un solo confronto vettoriale.
    La ricorsione e' sui nodi dell'albero, che sono una manciata, non sui campioni.
    """
    previsioni = np.empty(len(X), dtype=int)

    def scendi(nodo, quali):
        if "foglia" in nodo:
            previsioni[quali] = nodo["foglia"]
            return
        a_sinistra = quali & (X[nodo["feature"]].to_numpy() <= nodo["soglia"])
        scendi(nodo["si"], a_sinistra)
        scendi(nodo["no"], quali & ~a_sinistra)

    scendi(radice, np.ones(len(X), dtype=bool))
    return pd.Series(previsioni, index=X.index)

albero = cresci(X, y)
stampa(albero)

n_attivita_distinte <= 22.500 ?   IG=1.0000   (12 campioni)
  si:
     => ABBANDONO = 1   (6 campioni)
  no:
     => ABBANDONO = 0   (6 campioni)


## 5. Prestazioni sul file `manuale.csv`

La consegna chiede di valutare il classificatore **sullo stesso file** su cui e' stato costruito.
E' una valutazione *in-sample*: dice se il modello ha imparato i dati che ha visto, non se
generalizza. Al punto 8 lo mettiamo alla prova sui 7.035 studenti di `training.csv`.

In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

pred = predici(albero, X)

risultati = manuale.assign(previsto=pred, corretto=lambda d: d["ABBANDONO"] == d["previsto"])
print(risultati[[FEATURE, "ABBANDONO", "previsto", "corretto"]].sort_values(FEATURE).to_string())

print(f"\naccuratezza = {accuracy_score(y, pred):.4f}")
print(f"precision   = {precision_score(y, pred):.4f}")
print(f"recall      = {recall_score(y, pred):.4f}")
print(f"F1          = {f1_score(y, pred):.4f}")
print("\nconfusion matrix [righe = vero, colonne = previsto]:")
print(pd.DataFrame(confusion_matrix(y, pred), index=["vero 0", "vero 1"], columns=["prev 0", "prev 1"]))

        n_attivita_distinte  ABBANDONO  previsto  corretto
USERID                                                    
4344                      3          1         1      True
1123                      5          1         1      True
3637                      5          1         1      True
4200                      5          1         1      True
3071                      9          1         1      True
4413                     19          1         1      True
3969                     26          0         0      True
1714                     40          0         0      True
2535                     42          0         0      True
3058                     59          0         0      True
5557                     62          0         0      True
635                      73          0         0      True

accuratezza = 1.0000
precision   = 1.0000
recall      = 1.0000
F1          = 1.0000

confusion matrix [righe = vero, colonne = previsto]:
        prev 0  prev 1
vero 0      

## 6. Verifica con le API di scikit-learn

La consegna chiede di implementare il classificatore *"utilizzando eventualmente delle API"*.
Avendo gia' scritto la nostra implementazione, usiamo l'API come **controprova**: se il codice
implementa davvero ID3, `DecisionTreeClassifier(criterion="entropy")` deve scegliere la stessa
feature, la stessa soglia e produrre le stesse 12 predizioni.

In [8]:
from sklearn.tree import DecisionTreeClassifier, export_text

albero_sk = DecisionTreeClassifier(criterion="entropy", random_state=SEME).fit(X, y)
print(export_text(albero_sk, feature_names=list(X.columns)))

pred_sk = albero_sk.predict(X)
print("stessa feature alla radice:", X.columns[albero_sk.tree_.feature[0]] == FEATURE)
print("stessa soglia            :", bool(np.isclose(albero_sk.tree_.threshold[0], SOGLIA)))
print("stesse 12 predizioni     :", bool((pred_sk == pred.to_numpy()).all()))
print("numero di foglie         :", albero_sk.get_n_leaves(), "(come il nostro albero)")

|--- n_attivita_distinte <= 22.50
|   |--- class: 1
|--- n_attivita_distinte >  22.50
|   |--- class: 0

stessa feature alla radice: True
stessa soglia            : True
stesse 12 predizioni     : True
numero di foglie         : 2 (come il nostro albero)


## 7. Controprova: togliamo la feature dominante

Dal Task 1 sappiamo che le prime sei feature misurano la stessa quantita'. Se il proxy migliore non
ci fosse, l'albero troverebbe la stessa struttura?

In [9]:
X_ridotto = X.drop(columns=FEATURE)

print("classifica delle feature alla radice, senza", FEATURE, ":")
print(classifica_feature(X_ridotto, y).round(4).to_string(index=False))

albero_ridotto = cresci(X_ridotto, y)
print("\nalbero ottenuto:")
stampa(albero_ridotto)

pred_ridotto = predici(albero_ridotto, X_ridotto)
print(f"\naccuratezza sul file manuale = {accuracy_score(y, pred_ridotto):.4f}")

# quante feature pareggiano allo split successivo, dove i campioni sono solo 7?
maschera = X_ridotto[albero_ridotto["feature"]] > albero_ridotto["soglia"]
figlio, y_figlio = X_ridotto[maschera], y[maschera]
print(f"\nclassifica nel nodo di destra ({len(figlio)} campioni), dove avviene il secondo split:")
print(classifica_feature(figlio, y_figlio).round(4).to_string(index=False))

for nome, cl in [("radice", classifica_feature(X_ridotto, y)), ("nodo di destra", classifica_feature(figlio, y_figlio))]:
    massimo = cl["information_gain"].max()
    pari = cl[np.isclose(cl["information_gain"], massimo)]["feature"].tolist()
    print(f"\n{nome}: guadagno massimo {massimo:.4f}, ottenuto da {len(pari)} feature -> {pari}")

classifica delle feature alla radice, senza n_attivita_distinte :
        feature  soglia_migliore  information_gain
       n_azioni           31.500            0.6549
n_giorni_attivi            3.500            0.6549
  durata_giorni            4.813            0.6549
 feature0_media           -0.179            0.6549
 feature3_media           -0.062            0.6549
 feature2_media           -0.039            0.4591
 feature1_media            0.082            0.1957

albero ottenuto:
n_azioni <= 31.500 ?   IG=0.6549   (12 campioni)
  si:
     => ABBANDONO = 1   (5 campioni)
  no:
     feature0_media <= -0.179 ?   IG=0.5917   (7 campioni)
       si:
          => ABBANDONO = 1   (1 campioni)
       no:
          => ABBANDONO = 0   (6 campioni)

accuratezza sul file manuale = 1.0000

classifica nel nodo di destra (7 campioni), dove avviene il secondo split:
        feature  soglia_migliore  information_gain
 feature0_media          -0.1790            0.5917
 feature2_media           0.

Si': due livelli, ancora 12 su 12.

**Fregatura.** Il criterio quasi non riesce a scegliere: alla radice **5 feature su 7** pareggiano,
nel nodo di destra **2 su 7**. Quella che finisce nell'albero e' la prima nell'ordine delle colonne,
non la migliore.

## 8. Prestazioni su `training.csv`

Applichiamo l'albero **senza riaddestrarlo** ai 7.035 studenti.

In [10]:
training = pd.read_csv("../data/training.csv").set_index("USERID")
y_tr = training["ABBANDONO"]
X_tr = training.drop(columns="ABBANDONO")

pred_tr = predici(albero, X_tr)
print(f"studenti valutati: {len(y_tr)}  (positivi {y_tr.mean():.1%})")
print(f"accuratezza = {accuracy_score(y_tr, pred_tr):.4f}   F1 = {f1_score(y_tr, pred_tr):.4f}")
print("\nconfusion matrix:")
print(pd.DataFrame(confusion_matrix(y_tr, pred_tr), index=["vero 0", "vero 1"], columns=["prev 0", "prev 1"]))

# La soglia scelta su 12 campioni sarebbe stata la stessa con 7.035? Le 59 soglie si provano
# insieme: `colonna[:, None] <= soglie` e' la matrice 7.035 x 59 di tutte le predizioni, e il
# confronto con l'etichetta da' le 59 accuratezze con una sola media di colonna.
soglie = np.arange(2, 61)
previsto = X_tr[FEATURE].to_numpy()[:, None] <= soglie
scan = pd.DataFrame({"soglia": soglie,
                     "accuratezza": (previsto == (y_tr.to_numpy()[:, None] == 1)).mean(axis=0)})
migliore = scan.loc[scan["accuratezza"].idxmax()]
print(f"\nsoglia scelta sui 12 campioni       : {SOGLIA}   -> accuratezza {accuracy_score(y_tr, pred_tr):.4f}")
print(f"soglia ottima stimata su 7.035      : {migliore['soglia']:.0f}   -> accuratezza {migliore['accuratezza']:.4f}")

studenti valutati: 7035  (positivi 57.7%)
accuratezza = 0.7734   F1 = 0.8032

confusion matrix:
        prev 0  prev 1
vero 0    2189     786
vero 1     808    3252

soglia scelta sui 12 campioni       : 22.5   -> accuratezza 0.7734
soglia ottima stimata su 7.035      : 28   -> accuratezza 0.7808


## 9. Limiti

- **In-sample.** Dal 100% al 77,3%: feature e soglia scelte guardando gli stessi 12 campioni.
- **Ma il taglio regge.** La soglia ottima stimata su 7.035 studenti e' 28, e rende 0,7808 contro
  0,7734: 0,74 punti. I 12 campioni non sbagliano *dove* tagliare, sovrastimano quanto sia pulito.
- **Capacita' bassissima.** Ogni studente con piu' di 22 attivita' riceve la stessa risposta. Il
  77,3% e' il tetto di questa forma, non un errore di addestramento.
- **Tautologia.** Vale su una finestra iniziale (appendice del Task 1).